# Ridge regression - manual primal/dual workflow
Based on `polynomial regression.ipynb` and the ridge formulas in Tutorials 5-6. All imported libraries and numerical functions already appear in the tutorials.

**How to use:** edit `X`, `y`, `order`, positive `lmd`, and `X_test`, then **Run All**. One sample per row. Set `order = 1` for linear ridge or a higher order for polynomial ridge. The notebook displays the selected formula, named weights, training MSE, and predictions.


In [1]:
import numpy as np
from numpy.linalg import inv
from sklearn.preprocessing import PolynomialFeatures


## Training inputs
Enter **raw features** in `X`, without an added column of ones. `PolynomialFeatures` will add the intercept.

- `X`: m samples by p raw features.
- `y`: m rows; use a column for one output or multiple columns for multiple outputs. A 1-D target vector is also accepted.

The example below is copied from the polynomial regression notebook.


In [2]:
# EDIT INPUTS: replace these training features and targets.
X = np.array([[3,3.5], 
              [9,2.3],
              [7,5],
              [2,7]])
y = np.array([[-1],
              [1],
              [-1],
              [-1]])


## Polynomial terms and ridge strength
Let p be the number of raw features and n be the polynomial order. The number of terms, including the intercept and every power/interaction up to order n, is:

$$
\boxed{d = \binom{p+n}{n} = \frac{(p+n)!}{p!\,n!}}
$$

For p = 2 and n = 2, d = 6, with terms:

$$
1,\quad x_1,\quad x_2,\quad x_1^2,\quad x_1x_2,\quad x_2^2.
$$

The expanded matrix P (`X_poly`) has shape m by d. Each output has d coefficients. `PolynomialFeatures` includes the constant feature by default.

**Set `lmd` to the positive lambda given in the question.** The default `0.1` below is an illustrative value. Larger lambda applies a stronger penalty to large weights. Lambda zero removes the ridge penalty; ordinary inverses then require the rank checks shown in the polynomial notebook.


In [3]:
# EDIT INPUTS
order = 2
lmd = 0.1  # Demonstration only: replace with the question's lambda.
assert lmd > 0, "Use a positive lambda for ridge regression."

poly = PolynomialFeatures(degree=order)
X_poly = poly.fit_transform(X)
print("Polynomial matrix shape:", X_poly.shape)


Polynomial matrix shape: (4, 6)


## Ridge objective and training formulas
For one target column, ridge minimizes:

$$
\boxed{\|Pw-y\|_2^2 + \lambda\|w\|_2^2},\qquad \lambda > 0.
$$

The same formulas apply to each target column for multiple outputs. This is an **unnormalized sum of squared errors**, plus the penalty. Following Tutorials 5-6, **all coefficients, including the intercept, are penalized**.

### Primal form: rows >= columns

$$
\boxed{w = (P^TP + \lambda I_d)^{-1}P^Ty}
$$

Invert a d by d matrix. This is the regularized counterpart of the left-inverse formula.

### Dual form: rows < columns

$$
\boxed{w = P^T(PP^T + \lambda I_m)^{-1}y}
$$

Invert an m by m matrix. This is the regularized counterpart of the right-inverse formula.

The two ridge forms give the same weights mathematically. The manual `if/else` chooses the smaller matrix to invert. Positive lambda makes both penalized matrices invertible even if P is rank deficient; ridge does not require full row/column rank. Unlike an unregularized right-inverse fit, a ridge fit need not reproduce the training targets exactly.


In [4]:
# Keep targets as columns, preventing accidental broadcasting.
y = np.array(y, dtype=float)
if len(y.shape) == 1:
    y = y.reshape(-1, 1)
assert len(y.shape) == 2 and y.shape[0] == X_poly.shape[0], "X and y must have the same number of rows."
assert lmd > 0, "Use a positive lambda for ridge regression."

# MANUAL IF/ELSE: choose the smaller ridge matrix to invert.
m, d = X_poly.shape
if m >= d:
    # PRIMAL: identity size = number of polynomial columns.
    w = inv(X_poly.T @ X_poly + lmd * np.identity(d)) @ X_poly.T @ y
    print("Using PRIMAL ridge (regularized left-inverse formula)")
else:
    # DUAL: identity size = number of training samples.
    w = X_poly.T @ inv(X_poly @ X_poly.T + lmd * np.identity(m)) @ y
    print("Using DUAL ridge (regularized right-inverse formula)")

print("Lambda:", lmd)
print("Polynomial matrix shape/rank:", X_poly.shape, np.linalg.matrix_rank(X_poly))
print("Weight matrix shape:", w.shape)


Using DUAL ridge (regularized right-inverse formula)
Lambda: 0.1
Polynomial matrix shape/rank: (4, 6) 4
Weight matrix shape: (6, 1)


## Display the weights
For two features and order 2, the model is:

$$
\hat y = w_0 + w_1x_1 + w_2x_2 + w_3x_1^2 + w_4x_1x_2 + w_5x_2^2.
$$

The term labelled `1` is the intercept. Names start at `x1` to match the question. The display below adapts to the number of raw features, the order, and multiple output columns. Keep full-precision weights for predictions; round only the displayed answers.


In [5]:
raw_names = [f"x{i + 1}" for i in range(X.shape[1])]
terms = poly.get_feature_names_out(raw_names)
for output in range(w.shape[1]):
    if w.shape[1] > 1:
        print(f"\nOutput {output + 1}:")
    for i in range(len(terms)):
        print(f"{terms[i]}: {w[i, output]:.6f}")


1: -0.076600
x1: -0.155967
x2: -0.164454
x1^2: 0.047903
x1 x2: -0.055160
x2^2: 0.022835


## Training predictions and MSE
For one output:

$$
\hat y = Pw,\qquad
\mathrm{MSE} = \frac{1}{m}\sum_{i=1}^{m}(\hat y_i-y_i)^2.
$$

MSE measures prediction error; it does **not** include the ridge penalty. With multiple outputs, the code averages squared errors over both samples and outputs. Targets and predictions must have the same shape before subtraction.


In [6]:
y_train_pred = X_poly @ w
MSE_train = ((y_train_pred - y) ** 2).mean()
print("Training predictions:\n", y_train_pred)
print("Training MSE:", MSE_train)


Training predictions:
 [[-0.98841284]
 [ 1.00060199]
 [-1.00310488]
 [-1.00142426]]
Training MSE: 3.6573347597214924e-05


## Predict new samples
Enter raw test features in `X_test`, using the same feature order and count as training. For one sample with two features, use `[[10, 4.5]]`.

$$
\boxed{\hat y_{\mathrm{test}} = \phi(X_{\mathrm{test}})w}
$$

Use `poly.transform(X_test)` to apply the existing training mapping. Do not add another bias column or change the order at prediction time.

The output is a continuous regression value. If the question instead asks for binary classification with labels -1/+1, apply `np.sign(y_pred)` to obtain the class, using the question's tie rule for a zero score.


In [7]:
# EDIT INPUT: one new sample per row.
X_test = np.array([[10, 4.5]])
X_test_poly = poly.transform(X_test)
y_pred = X_test_poly @ w

print("Predicted y:\n", y_pred)
if y_pred.shape == (1, 1):
    print(f"Predicted y (4 d.p.): {y_pred[0, 0]:.4f}")
# If binary classification is requested: print(np.sign(y_pred))


Predicted y:
 [[0.39422206]]
Predicted y (4 d.p.): 0.3942
